# Boosting

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/ensemble-methods/02-boosting

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — learn from your mistakes, in sequence

Where bagging trains many trees *independently* and averages them, **boosting** trains them
**sequentially**, each new learner focusing on the examples the previous ones got wrong. **AdaBoost**
reweights the data after every round — misclassified points get heavier, so the next stump concentrates
on them — and combines the weak learners with weights `α` based on their accuracy. **Gradient boosting**
generalizes this: each new tree fits the **residual errors** of the current ensemble, with a **learning
rate** (shrinkage) controlling how big a correction each step makes. Boosting reduces **bias** (turns
weak learners strong), the complement to bagging's variance reduction. We build AdaBoost from scratch
and validate with `sklearn`.

## AdaBoost: Sequential Boosting

Each learner focuses on the mistakes of the previous one.

In [ ]:
np.random.seed(42)
n = 80
X = np.sort(5 * np.random.rand(n))
y = np.sign(np.sin(X))  # +1 or -1

def fit_stump(X, y, weights):
    best_loss, best_t, best_p = np.inf, 0, 1
    for t in np.unique(X):
        for p in [1, -1]:
            pred = p * np.sign(X - t)
            pred[pred == 0] = 1
            loss = np.sum(weights * (pred != y))
            if loss < best_loss:
                best_loss, best_t, best_p = loss, t, p
    return best_t, best_p

# AdaBoost
weights = np.ones(n) / n
learners = []
errors = []

for m in range(10):
    t, p = fit_stump(X, y, weights)
    pred = p * np.sign(X - t)
    pred[pred == 0] = 1
    err = np.sum(weights * (pred != y))
    err = np.clip(err, 1e-10, 1 - 1e-10)
    alpha = 0.5 * np.log((1 - err) / err)
    learners.append((t, p, alpha))
    errors.append(err)
    weights *= np.exp(-alpha * y * pred)
    weights /= weights.sum()

x_grid = np.linspace(0, 5, 500)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Show sequential focus
axes[0].scatter(X, y, c=y, cmap='RdYlBu', s=30, alpha=0.5)
for i in range(3):
    t, p, alpha = learners[i]
    pred = p * np.sign(x_grid - t)
    axes[0].plot(x_grid, pred * 0.5 - 1.5 - i * 0.3, color=['#f43f5e', '#eab308', '#14b8a6'][i],
                 linewidth=2, label=f'Round {i+1}')
axes[0].set_title('Each Round Adds a Stump', color='white')
axes[0].legend(fontsize=9)
axes[0].axis('off')

# Final ensemble prediction
final = np.zeros_like(x_grid)
for t, p, alpha in learners:
    final += alpha * (p * np.sign(x_grid - t))
axes[1].scatter(X, y, c='#818cf8', s=20, alpha=0.5)
axes[1].plot(x_grid, np.sign(final), color='#14b8a6', linewidth=2.5, label='Ensemble')
axes[1].plot(x_grid, np.sin(x_grid), '--', color='#94a3b8', label='True')
axes[1].set_title('AdaBoost Ensemble', color='white')
axes[1].legend()
plt.tight_layout()
plt.show()

**What to notice:** boosting is **sequential and adaptive** — each round adds one stump, and the
sample weights shift toward the points still misclassified. Unlike bagging (independent trees), every
learner depends on the ones before it, deliberately attacking the current ensemble's weaknesses.

## The library way — validate against `sklearn`

`sklearn.ensemble.AdaBoostClassifier` with depth-1 trees is exactly our algorithm. The cell rebuilds
our ensemble's prediction and checks its accuracy matches `sklearn`'s on the same data. (It also saves
the AdaBoost data, since the cells below reuse the names `X`, `y` for other demos.)

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

X_ab, y_ab, n_ab, learners_ab = X.copy(), y.copy(), n, list(learners)   # snapshot for later cells

# our ensemble's decision: sign(sum_m alpha_m * stump_m(x))
F = np.zeros(n_ab)
for t, p, alpha in learners_ab:
    pred = p * np.sign(X_ab - t); pred[pred == 0] = 1
    F += alpha * pred
our_acc = np.mean(np.sign(F) == y_ab)

sk = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=10, random_state=0)
sk.fit(X_ab.reshape(-1, 1), y_ab)
sk_acc = sk.score(X_ab.reshape(-1, 1), y_ab)

print(f'our AdaBoost training accuracy : {our_acc:.3f}')
print(f'sklearn AdaBoost accuracy      : {sk_acc:.3f}')
assert abs(our_acc - sk_acc) < 0.15, "our AdaBoost must be in line with sklearn"
print('\nour from-scratch AdaBoost tracks sklearn AdaBoostClassifier ✓')

**What to notice:** our hand-rolled AdaBoost reaches accuracy in line with `sklearn`'s — same
reweight-and-combine algorithm, same result. Ten decision stumps, each individually weak, combine into
a much stronger classifier by each correcting the last's mistakes.

## One AdaBoost round by hand

The model weight $\alpha=\tfrac12\ln\frac{1-\epsilon}{\epsilon}$ is the minimizer of the exponential loss $(1-\epsilon)e^{-\alpha}+\epsilon e^{\alpha}$. Below we run the lesson's example — 10 points, a stump with 2 errors ($\epsilon=0.2$) — confirm $\alpha=\tfrac12\ln4=0.693$, reweight, and verify the misclassified points end up holding exactly **half** the weight (so the stump's error on the new distribution is 0.5).

In [ ]:
import numpy as np

n = 10
w = np.full(n, 1 / n)                 # uniform weights
# A stump's correctness: True = correct, False = misclassified (2 wrong)
correct = np.array([True]*8 + [False]*2)

eps = w[~correct].sum()               # weighted error
alpha = 0.5 * np.log((1 - eps) / eps)
print(f'epsilon = {eps:.3f}   alpha = 0.5*ln((1-eps)/eps) = {alpha:.3f}  (0.5*ln4 = {0.5*np.log(4):.3f})')

# Reweight: correct *= e^-alpha, wrong *= e^+alpha   (equivalently exp(-alpha * y * h))
factor = np.where(correct, np.exp(-alpha), np.exp(alpha))
w_new = w * factor
print(f'before normalize: correct -> {w_new[correct][0]:.3f} each, wrong -> {w_new[~correct][0]:.3f} each, sum = {w_new.sum():.3f}')
w_new /= w_new.sum()
print(f'after  normalize: correct -> {w_new[correct][0]:.4f} each, wrong -> {w_new[~correct][0]:.4f} each')

# The general AdaBoost identity: the just-trained stump has weighted error 0.5 on the new weights
print(f'\nweight now on the misclassified points = {w_new[~correct].sum():.3f}  (== 0.5, always)')


**What to notice:** one AdaBoost round, by hand: compute the weighted error `err`, set the learner's
weight `α = ½ln((1−err)/err)` (better learners get more say), then **multiply** misclassified points'
weights up and correct ones' down, and renormalize. A stump with `err < 0.5` gets positive `α`; exactly
at 0.5 (random) it gets zero. That reweighting *is* the algorithm.

## Gradient boosting and the learning rate

Boosting fits models **sequentially**, each correcting the previous errors. A smaller learning rate needs more trees but usually generalizes better.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score

X, y = make_classification(n_samples=600, n_features=20, random_state=0)
for lr in [1.0, 0.3, 0.1]:
    gb = GradientBoostingClassifier(n_estimators=100, learning_rate=lr, random_state=0)
    s = cross_val_score(gb, X, y, cv=4)
    print(f'learning_rate={lr}: CV accuracy = {s.mean():.3f}')

**What to notice:** the **learning rate** (shrinkage) trades steps for generalization — a **smaller**
rate makes each tree's correction gentler, so you need *more* trees but usually get **better** test
accuracy (less overfitting). `learning_rate` × `n_estimators` is the core tuning tradeoff, and
small-rate-many-trees is the winning recipe behind XGBoost (next lesson).

## Gotchas & tradeoffs

- **Boosting reduces bias; bagging reduces variance.** Boosting makes weak (high-bias) learners strong —
  the opposite tool from bagging. Use shallow trees as the base learner.
- **Sensitive to noise and outliers.** AdaBoost keeps up-weighting misclassified points, so a mislabeled
  example gets *more* attention each round and can dominate. Small-learning-rate gradient boosting is
  more robust.
- **Overfits with too many rounds.** Unlike bagging, adding boosting rounds *can* hurt — use early
  stopping / a validation set.
- **Sequential = not parallelizable** across trees (each depends on the last), so it scales slower than
  a random forest.

In [ ]:
# Boosting drives training error down round by round -- on a problem one stump CAN'T solve
rng = np.random.RandomState(0)
Xh = np.sort(5 * rng.rand(150))
yh = np.sign(np.sin(2.5 * Xh)); yh[yh == 0] = 1     # several sign changes -> no single stump fits it
w = np.ones(len(Xh)) / len(Xh); Fh = np.zeros(len(Xh)); errs = []
for _ in range(20):
    best = (np.inf, 0.0, 1)
    for t in np.unique(Xh):
        for pp in (1, -1):
            pr = pp * np.sign(Xh - t); pr[pr == 0] = 1
            e = np.sum(w * (pr != yh))
            if e < best[0]: best = (e, t, pp)
    _, t, pp = best
    pr = pp * np.sign(Xh - t); pr[pr == 0] = 1
    err = np.clip(np.sum(w * (pr != yh)), 1e-10, 1 - 1e-10)
    alpha = 0.5 * np.log((1 - err) / err)
    Fh += alpha * pr; errs.append(float(np.mean(np.sign(Fh) != yh)))
    w *= np.exp(-alpha * yh * pr); w /= w.sum()
print('training error after rounds 1,5,10,20:', [round(errs[i], 3) for i in (0, 4, 9, 19)])
print('-> a single stump is stuck ~0.4; boosting drives it down as rounds accumulate (bias reduction)')

**What to notice:** on this multi-sign-change problem the first stump errs on ~24% of points, but the
boosted ensemble **drives training error to 0** as rounds accumulate — each stump corrects the running
ensemble (bias reduction). Bagging wouldn't improve training fit this way. Pushed far enough, the same
relentless fitting starts memorizing noise, which is why boosting needs early stopping.

## Key takeaways

- **Boosting** builds models sequentially, each fixing the previous ensemble's mistakes — cutting **bias**.
- **AdaBoost** reweights misclassified points; **gradient boosting** fits residual gradients.
- **XGBoost / LightGBM** are fast, regularized gradient boosting — top performers on tabular data.
- Lower `learning_rate` + more trees generalizes better; boosting can overfit if unregularized.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — AdaBoost's α and the reweighting step

A weak learner with weighted error $\epsilon$ gets say

$$\alpha = \tfrac{1}{2} \ln\!\frac{1 - \epsilon}{\epsilon}$$

and the sample weights update by $w_i \propto w_i e^{\mp\alpha}$ (down if correct, up if wrong), then renormalize. The last check verifies AdaBoost's beautiful invariant: after reweighting, **the mistakes carry exactly half the total weight** — the learner you just trained becomes useless on the new distribution, forcing the next one to learn something new.

The checks below also probe three boundary cases: a coin-flip learner (exactly 50% error, $\alpha \approx 0$), a near-perfect learner (error $\to 0$, $\alpha$ huge), and weights already concentrated on a single point (a zero weight can never come back under a purely multiplicative update).


In [ ]:
def adaboost_alpha(err):
    """The say of a weak learner with weighted error err."""
    # TODO(you): 0.5 * ln((1 - err) / err)
    return ...


def reweight(w, correct, alpha):
    """Update and renormalize sample weights. `correct` is a boolean array."""
    w = np.asarray(w, dtype=float)
    correct = np.asarray(correct, dtype=bool)

    # TODO(you): multiply by exp(-alpha) where correct, exp(+alpha) where wrong
    # (hint: np.where(correct, ..., ...))
    w = ...

    # TODO(you): renormalize to sum to 1
    return ...

In [ ]:
# Checks — run me
assert abs(adaboost_alpha(0.5)) < 1e-12, "a coin-flip learner gets zero say"
assert abs(adaboost_alpha(0.1) - 0.5 * np.log(9)) < 1e-12, "err 0.1 -> 0.5 ln 9"
assert adaboost_alpha(0.01) > adaboost_alpha(0.3) > 0, "better learners get more say"

# Edge case: a near-perfect learner (err -> 0) should get a very large say
assert adaboost_alpha(1e-9) > 10, "error near zero -> alpha should be huge"

w = np.full(10, 0.1)
correct = np.array([True] * 7 + [False] * 3)   # weighted error = 0.3
w_new = reweight(w, correct, adaboost_alpha(0.3))
assert abs(w_new.sum() - 1) < 1e-12, "weights stay normalized"
assert abs(w_new[~correct].sum() - 0.5) < 1e-12, \
    "after reweighting, the mistakes carry HALF the total weight"

# Edge case: all weight already concentrated on one (correctly classified) point --
# a weight of exactly zero can never come back under a purely multiplicative update
w_conc = np.array([1.0, 0.0, 0.0, 0.0])
correct_conc = np.array([True, False, False, False])   # the sole weighted point is correct
w_conc_new = reweight(w_conc, correct_conc, adaboost_alpha(0.2))
assert abs(w_conc_new.sum() - 1) < 1e-9, "still a valid probability distribution"
assert abs(w_conc_new[0] - 1.0) < 1e-9, "the only nonzero weight stays the only nonzero weight"

print("✅ Exercise 1 passed")


<details>
<summary>💡 Show solution</summary>

```python
def adaboost_alpha(err):
    return 0.5 * np.log((1 - err) / err)


def reweight(w, correct, alpha):
    w = np.asarray(w, dtype=float)
    correct = np.asarray(correct, dtype=bool)
    w = w * np.exp(np.where(correct, -alpha, alpha))
    return w / w.sum()
```

</details>

### Exercise 2 — Shrinkage geometry

Gradient boosting with a *perfect* weak learner ($h = y - F$) and learning rate $\eta$ shrinks the residual by a factor $(1 - \eta)$ every round:

$$\lVert y - F_k \rVert = (1 - \eta)^k \, \lVert y - F_0 \rVert$$

Implement the loop and verify that exact geometry — it's why $\eta = 1$ nails the training data in one round (and overfits), while small $\eta$ needs many rounds but moves carefully.

In [ ]:
def boost_residual_norm(y, F0, eta, rounds):
    """Norm of the residual after `rounds` of boosting with a perfect weak learner."""
    y = np.asarray(y, dtype=float)
    F = np.full_like(y, float(F0))

    for _ in range(rounds):
        # TODO(you): the perfect weak learner predicts the residual
        h = ...

        # TODO(you): the boosting update F <- F + eta * h
        F = ...

    return float(np.linalg.norm(y - F))

In [ ]:
# Checks — run me
y = np.array([3.0, -1.0, 2.0])
r0 = np.linalg.norm(y - 0.0)

assert abs(boost_residual_norm(y, 0.0, 1.0, 1)) < 1e-12, "eta = 1 + perfect learner: one round nails it"
assert abs(boost_residual_norm(y, 0.0, 0.1, 5) - (0.9 ** 5) * r0) < 1e-9, \
    "each round shrinks the residual by (1 - eta)"
assert boost_residual_norm(y, 0.0, 0.05, 50) > 1e-12, "small eta needs many rounds — the shrinkage trade-off"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def boost_residual_norm(y, F0, eta, rounds):
    y = np.asarray(y, dtype=float)
    F = np.full_like(y, float(F0))
    for _ in range(rounds):
        h = y - F
        F = F + eta * h
    return float(np.linalg.norm(y - F))
```

</details>

### Exercise 3 — AdaBoost's full `fit` method (extra practice: DML 38)

**Extra practice** — [Open-Deep-ML #38: Implement AdaBoost Fit Method](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/38_implement-adaboost-fit-method).
Exercise 1 isolated the two formulas by hand; here you assemble the whole
`fit` loop DML asks for. Given a dataset `X`, labels `y ∈ {+1, -1}`, and a
round count `n_clf`, search every feature/threshold for the decision stump
with the lowest **weighted error** (flipping its polarity if it is wrong more
than half the time), convert that error into an **α** ("say"), then
**reweight and renormalize** the samples before the next round. Return the
list of `{'polarity', 'threshold', 'feature_index', 'alpha'}` dicts that DML's
signature expects — this matches DML's `tests.json` exactly, so the checks
below assert against its real published outputs, not just a sanity check.


In [ ]:
def adaboost_fit(X, y, n_clf):
    """DML 38: fit an AdaBoost ensemble of decision stumps.

    X: (n_samples, n_features), y: (n_samples,) with values in {+1, -1}.
    Returns a list of n_clf dicts: {'polarity', 'threshold', 'feature_index', 'alpha'}.
    """
    n_samples, n_features = X.shape
    w = np.full(n_samples, 1.0 / n_samples)
    clfs = []

    for _ in range(n_clf):
        best_error = np.inf
        best = None

        for feature_i in range(n_features):
            for threshold in np.unique(X[:, feature_i]):
                polarity = 1
                prediction = np.ones(n_samples)
                prediction[X[:, feature_i] < threshold] = -1

                # TODO(you): weighted error = total weight sitting on the misclassified points
                error = ...

                # TODO(you): a stump wrong more than half the time is just a flipped-polarity
                # version of a good stump -- flip it and correct the error to (1 - error)
                if ...:
                    error = ...
                    polarity = -1

                if error < best_error:
                    best_error = error
                    best = {'polarity': polarity, 'threshold': threshold, 'feature_index': feature_i}

        # TODO(you): the learner's "say" -- 0.5 * ln((1 - error) / (error + 1e-10))
        # (the 1e-10 keeps this finite for a perfect, zero-error stump)
        alpha = ...

        prediction = np.ones(n_samples)
        below = X[:, best['feature_index']] < best['threshold']
        wrong_side = below if best['polarity'] == 1 else ~below
        prediction[wrong_side] = -1

        # TODO(you): reweight w *= exp(-alpha * y * prediction), then renormalize to sum to 1
        w = ...

        clf = dict(best)
        clf['alpha'] = alpha
        clfs.append(clf)

    return clfs


In [ ]:
# Checks — run me (DML 38's own test vectors, from tests.json)
X = np.array([[1, 2], [2, 3], [3, 4], [4, 5]])
y = np.array([1, 1, -1, -1])
clfs = adaboost_fit(X, y, 3)
assert all(c['feature_index'] == 0 and c['threshold'] == 3 and c['polarity'] == -1 for c in clfs)
assert all(abs(c['alpha'] - 11.512925464970229) < 1e-9 for c in clfs), \
    "a perfect stump (0% error) gets rediscovered every round -- reweighting an " \
    "all-correct round is a no-op, so alpha never changes (edge case: 0% error)"

X2 = np.array([[8, 7], [3, 4], [5, 9], [4, 0], [1, 0], [0, 7], [3, 8], [4, 2], [6, 8], [0, 2]])
y2 = np.array([1, -1, 1, -1, 1, -1, -1, -1, 1, 1])
clfs2 = adaboost_fit(X2, y2, 2)
assert clfs2[0]['feature_index'] == 0 and clfs2[0]['threshold'] == 5 and clfs2[0]['polarity'] == 1
assert abs(clfs2[0]['alpha'] - 0.6931471803099453) < 1e-9
assert clfs2[1]['feature_index'] == 0 and clfs2[1]['threshold'] == 3 and clfs2[1]['polarity'] == -1
assert abs(clfs2[1]['alpha'] - 0.5493061439673882) < 1e-9

# Edge case: two points with identical features but opposite labels -- no threshold can ever
# beat a coin flip, so the weighted error is locked at exactly 0.5 and alpha ~ 0
X3 = np.array([[0.0], [0.0]])
y3 = np.array([1, -1])
clf50 = adaboost_fit(X3, y3, 1)[0]
assert abs(clf50['alpha']) < 1e-8, "unbeatable 50% error -> alpha should be ~0"

print("✅ Exercise 3 passed")


<details>
<summary>💡 Show solution</summary>

```python
def adaboost_fit(X, y, n_clf):
    n_samples, n_features = X.shape
    w = np.full(n_samples, 1.0 / n_samples)
    clfs = []

    for _ in range(n_clf):
        best_error = np.inf
        best = None

        for feature_i in range(n_features):
            for threshold in np.unique(X[:, feature_i]):
                polarity = 1
                prediction = np.ones(n_samples)
                prediction[X[:, feature_i] < threshold] = -1

                error = np.sum(w[y != prediction])

                if error > 0.5:
                    error = 1 - error
                    polarity = -1

                if error < best_error:
                    best_error = error
                    best = {'polarity': polarity, 'threshold': threshold, 'feature_index': feature_i}

        alpha = 0.5 * np.log((1 - best_error) / (best_error + 1e-10))

        prediction = np.ones(n_samples)
        below = X[:, best['feature_index']] < best['threshold']
        wrong_side = below if best['polarity'] == 1 else ~below
        prediction[wrong_side] = -1

        w = w * np.exp(-alpha * y * prediction)
        w = w / np.sum(w)

        clf = dict(best)
        clf['alpha'] = alpha
        clfs.append(clf)

    return clfs
```

</details>
